In [2]:
import pandas as pd

# Φόρτωση του αρχείου (ανεβασμένο στο Colab)
file_path = "/content/Pre-term-labour-Data-ready-for-ml-pipeline_final.xlsx"

# Διάβασε το Excel
df = pd.read_excel(file_path)

# Μέτρησε τις ηλικίες κάτω των 35
count = df[df["Maternal age"] < 35].shape[0]
print(f"Το πλήθος των ηλικιών κάτω των 35 είναι: {count}")

# Μέτρησε τις ηλικίες άνω των 35
count = df[df["Maternal age"] >= 35].shape[0]
print(f"Το πλήθος των ηλικιών άνω των 35 είναι: {count}")

# Μέτρησε τις ηλικίες κάτω των 40
count = df[df["Maternal age"] < 40].shape[0]
print(f"Το πλήθος των ηλικιών κάτω των 40 είναι: {count}")

# Μέτρησε τις ηλικίες άνω των 40
count = df[df["Maternal age"] >= 40].shape[0]
print(f"Το πλήθος των ηλικιών άνω των 40 είναι: {count}")

# Φιλτράρισμα για ηλικίες < 35 και preterm birth = 1
count = df[(df["Maternal age"] < 35) & (df["Preterm_birth"] == 1)].shape[0]
print(f"Το πλήθος των ηλικιών κάτω των 35 με preterm birth = 1 είναι: {count}")

# Φιλτράρισμα για ηλικίες >= 35 και preterm birth = 1
count = df[(df["Maternal age"] >= 35) & (df["Preterm_birth"] == 1)].shape[0]
print(f"Το πλήθος των ηλικιών άνω των 35 με preterm birth = 1 είναι: {count}")

Το πλήθος των ηλικιών κάτω των 35 είναι: 627
Το πλήθος των ηλικιών άνω των 35 είναι: 343
Το πλήθος των ηλικιών κάτω των 40 είναι: 898
Το πλήθος των ηλικιών άνω των 40 είναι: 72
Το πλήθος των ηλικιών κάτω των 35 με preterm birth = 1 είναι: 203
Το πλήθος των ηλικιών άνω των 35 με preterm birth = 1 είναι: 123


In [3]:
!pip install aif360
!pip install aif360 BlackBoxAuditing
import numpy as np
import pandas as pd
from aif360.datasets import StandardDataset
from aif360.metrics import BinaryLabelDatasetMetric
from aif360.datasets import BinaryLabelDataset
from aif360.algorithms.preprocessing import DisparateImpactRemover

df.head()

,Maternal age,GA,BW centile,UtA doppler,b-hcg,DVP,MCA doppler,Papp-A,Height,UA doppler,...,Placental location_high posterior with anterior paraplacenta,Placental location_high right,Placental location_ligh anterior with posterior paraplacenta,Placental location_low anterior,Placental location_low posterior,Placental location_low posterior with anterior paraplacenta,Placental location_low right,Placental location_previa,Single umbilical artery_0,Single umbilical artery_1
0,32.000000,24.285714,20.378457,1.010,0.98,5.1,2.03,0.90,166.0,1.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,33.000000,25.285714,16.129898,0.880,0.95,4.0,2.10,1.12,165.0,0.91,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,36.435616,32.571429,74.037300,0.675,1.30,3.6,1.87,0.50,175.0,0.79,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,42.000000,28.714286,71.680853,1.350,1.00,1.4,1.87,0.90,160.0,1.11,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,33.000000,29.285714,22.836726,0.640,1.13,4.1,2.02,1.12,163.0,1.08,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [ ]:
# Δημιουργία μίας δυαδικής μεταβλητής για την ηλικία (40 ως όριο)
df['Maternal age binary'] = (df['Maternal age'] >= 35).astype(int)

# Επιβεβαίωση του διαχωρισμού
print(df.groupby('Maternal age binary')['Preterm_birth'].mean())

dataset = StandardDataset(df,
                          label_name='Preterm_birth',
                          favorable_classes=[0],  # Θετικό αποτέλεσμα: Κανονικός τοκετός
                          protected_attribute_names=['Maternal age binary'],
                          privileged_classes=[[0]])  # προνομιούχο group: Μητέρες <40

# Νέος ορισμός των ομάδων
unprivileged_groups = [{'Maternal age binary': 1}]  # Μη προνομιούχο group (>=40)
privileged_groups = [{'Maternal age binary': 0}]    # Προνομιούχο group (<40)

# Υπολογισμός fairness metrics
# Υπολογισμός fairness metrics
metric = BinaryLabelDatasetMetric(dataset,
                                  unprivileged_groups=unprivileged_groups,
                                  privileged_groups=privileged_groups)

print("Bias Metrics for Maternal age:")
print(f"Disparate Impact: {metric.disparate_impact()}")
print(f"Statistical Parity Difference: {metric.statistical_parity_difference()}")

Maternal age binary
0    0.323764
1    0.358601
Name: Preterm_birth, dtype: float64
Bias Metrics for Maternal age:
Disparate Impact: 0.9484845150998404
Statistical Parity Difference: -0.03483662774747631


In [ ]:
print(df.groupby('History_of_PTD')['Preterm_birth'].mean())

# Ορισμός του dataset για AIF360
dataset = StandardDataset(df,
                          label_name='Preterm_birth',  # Στόχος (target)
                          favorable_classes=[0],  # Θετική έκβαση (πλήρης κύηση)
                          protected_attribute_names=['History_of_PTD'],  # Προστατευμένη μεταβλητή
                          privileged_classes=[[0]])  # Προνόμιο: Χωρίς ιστορικό πρόωρου τοκετού

# Ορισμός privileged/unprivileged groups
unprivileged_groups = [{'History_of_PTD': 1}]  # Γυναίκες με ιστορικό πρόωρου τοκετού
privileged_groups = [{'History_of_PTD': 0}]  # Γυναίκες χωρίς ιστορικό πρόωρου τοκετού

# Υπολογισμός fairness metrics
metric = BinaryLabelDatasetMetric(dataset,
                                  unprivileged_groups=unprivileged_groups,
                                  privileged_groups=privileged_groups)

print("Bias Metrics for History_of_PTD:")
print(f"Disparate Impact: {metric.disparate_impact()}")
print(f"Statistical Parity Difference: {metric.statistical_parity_difference()}")

History_of_PTD
0.0    0.317797
1.0    1.000000
Name: Preterm_birth, dtype: float64
Bias Metrics for History_of_PTD:
Disparate Impact: 0.0
Statistical Parity Difference: -0.6822033898305084


In [ ]:
# Δημιουργία κατηγοριών BMI
df['BMI_category'] = pd.cut(df['BMI'], bins=[0, 18.5, 24.9, 29.9, 100],
                            labels=['Underweight', 'Normal', 'Overweight', 'Obese'])

# Υπολογισμός μέσου ποσοστού πρόωρου τοκετού για κάθε κατηγορία
bmi_bias = df.groupby('BMI_category', observed=True)['Preterm_birth'].mean()

print(bmi_bias)

df['Obese'] = (df['BMI'] >= 30).astype(int)  # 1 = Παχυσαρκία, 0 = Μη-παχυσαρκία
df['BMI_category'] = df['BMI_category'].astype('category').cat.codes

dataset = StandardDataset(
    df,
    label_name='Preterm_birth',
    favorable_classes=[0],
    protected_attribute_names=['Obese'],
    privileged_classes=[[0]]
)

metric = BinaryLabelDatasetMetric(dataset, privileged_groups=[{'Obese': 0}], unprivileged_groups=[{'Obese': 1}])

print("Bias Metrics for BMI:")
print(f"Disparate Impact:", metric.disparate_impact())
print(f"Statistical Parity Difference:", metric.statistical_parity_difference())

BMI_category
Underweight    1.000000
Normal         0.329545
Overweight     0.303103
Obese          0.383803
Name: Preterm_birth, dtype: float64
Bias Metrics for BMI:
Disparate Impact: 0.9013033424427159
Statistical Parity Difference: -0.06747628628916358


In [ ]:
print(df.groupby('Cervical length')['Preterm_birth'].mean())

df['Short_Cervix'] = (df['Cervical length'] < 15).astype(int)  # 1 = Κοντός (<15mm), 0 = Κανονικός (≥15mm)

dataset = StandardDataset(
    df,
    label_name='Preterm_birth',  # Στόχος (label)
    favorable_classes=[0],  # 0 = Δεν υπήρξε πρόωρος τοκετός
    protected_attribute_names=['Short_Cervix'],  # Προστατευμένη μεταβλητή
    privileged_classes=[[0]]  # Προνομιούχο group (0 = Κανονικό μήκος τραχήλου)
)

metric = BinaryLabelDatasetMetric(dataset, privileged_groups=[{'Short_Cervix': 0}], unprivileged_groups=[{'Short_Cervix': 1}])

print("Bias Metrics for Cervical length:")
print(f"Disparate Impact:", metric.disparate_impact())
print(f"Statistical Parity Difference:", metric.statistical_parity_difference())

Cervical length
1.0     1.000000
5.0     1.000000
6.0     1.000000
7.0     1.000000
8.0     1.000000
9.0     1.000000
10.0    1.000000
11.0    1.000000
12.0    1.000000
13.0    1.000000
14.0    1.000000
15.0    0.866667
16.0    0.750000
17.0    0.800000
18.0    0.769231
19.0    0.333333
20.0    0.812500
20.9    1.000000
21.0    0.933333
22.0    0.970149
23.0    0.722222
24.0    0.892857
25.0    0.466667
26.0    0.583333
26.7    0.000000
27.0    0.363636
28.0    0.268293
29.0    0.000000
29.8    0.000000
30.0    0.000000
31.0    0.000000
32.0    0.000000
33.0    0.000000
34.0    0.000000
35.0    0.000000
36.0    0.000000
37.0    0.000000
38.0    0.000000
39.0    0.000000
40.0    0.000000
41.0    0.000000
42.0    0.000000
Name: Preterm_birth, dtype: float64
Bias Metrics for Cervical length:
Disparate Impact: 0.0
Statistical Parity Difference: -0.734321550741163


In [ ]:
print(df.groupby('Smoking')['Preterm_birth'].mean())

# Υπολογισμός metric
dataset = BinaryLabelDataset(df=df[['Smoking', 'Preterm_birth']].dropna(), label_names=['Preterm_birth'], protected_attribute_names=['Smoking'])

# Υπολογισμός metric
metric = BinaryLabelDatasetMetric(dataset, privileged_groups=[{'Smoking': 1}], unprivileged_groups=[{'Smoking': 0}])

# Εκτύπωση αποτελεσμάτων
print("Bias Metrics for Smoking:")
print(f"Disparate Impact: {metric.disparate_impact()}")
print(f"Statistical Parity Difference: {metric.statistical_parity_difference()}")

Smoking
0.0    0.327210
1.0    0.414141
Name: Preterm_birth, dtype: float64
Bias Metrics for Smoking:
Disparate Impact: 0.7900926885273446
Statistical Parity Difference: -0.0869313108119078


In [ ]:
import scipy.stats as stats

# Επιλογή των σχετικών μεταβλητών
df_subset = df[['Smoking', 'Preterm_birth']].dropna()  # Αφαίρεση τυχόν NaN τιμών

# Δημιουργία πίνακα σύγχυσης
contingency_table = pd.crosstab(df_subset['Smoking'], df_subset['Preterm_birth'])

# Εκτέλεση του Chi-Square Test
chi2, p, dof, expected = stats.chi2_contingency(contingency_table)

# Εκτύπωση αποτελεσμάτων
print(f"Chi-Square: {chi2:.4f}")
print(f"P-value: {p:.4f}")

# Απόφαση με βάση την p-value
if p < 0.05:
    print("Πιθανή στατιστικά σημαντική σχέση μεταξύ καπνίσματος και πρόωρου τοκετού!")
else:
    print("Δεν υπάρχει στατιστικά σημαντική σχέση.")

Chi-Square: 2.6338
P-value: 0.1046
Δεν υπάρχει στατιστικά σημαντική σχέση.


In [ ]:
# Δημιουργία πίνακα συχνοτήτων
contingency_table = pd.crosstab(df['Smoking'], df['Preterm_birth'])

# Εμφάνισε τον πίνακα συχνοτήτων
print(contingency_table)

from scipy.stats import fisher_exact

# Εκτέλεση Fisher's Exact Test
oddsratio, p_value = fisher_exact(contingency_table)

# Εμφάνιση του αποτελέσματος
print(f"Odd's ratio: {oddsratio}")
print(f"P-value: {p_value}")

Preterm_birth  0.0  1.0
Smoking                
0.0            586  285
1.0             58   41
Odd's ratio: 1.4534785238959467
P-value: 0.09218919336355383


In [ ]:
print(df.groupby('Conception_ART')['Preterm_birth'].mean())

# Υπολογισμός metric
dataset = BinaryLabelDataset(df=df[['Conception_ART', 'Preterm_birth']].dropna(), label_names=['Preterm_birth'], protected_attribute_names=['Conception_ART'])

# Υπολογισμός metric
metric = BinaryLabelDatasetMetric(dataset, privileged_groups=[{'Conception_ART': 1}], unprivileged_groups=[{'Conception_ART': 0}])

# Εκτύπωση αποτελεσμάτων
print("Bias Metrics for Concepion_ART:")
print(f"Disparate Impact: {metric.disparate_impact()}")
print(f"Statistical Parity Difference: {metric.statistical_parity_difference()}")

Conception_ART
0.0    0.335221
1.0    0.344828
Name: Preterm_birth, dtype: float64
Bias Metrics for Concepion_ART:
Disparate Impact: 0.9721404303510758
Statistical Parity Difference: -0.009606748154801448


In [ ]:
print(df.groupby('History_of_SGA_FGR')['Preterm_birth'].mean())

# Υπολογισμός metric
dataset = BinaryLabelDataset(df=df[['History_of_SGA_FGR', 'Preterm_birth']].dropna(), label_names=['Preterm_birth'], protected_attribute_names=['History_of_SGA_FGR'])

# Υπολογισμός metric
metric = BinaryLabelDatasetMetric(dataset, privileged_groups=[{'History_of_SGA_FGR': 1}], unprivileged_groups=[{'History_of_SGA_FGR': 0}])

# Εκτύπωση αποτελεσμάτων
print("Bias Metrics for History_of_SGA_FGR:")
print(f"Disparate Impact: {metric.disparate_impact()}")
print(f"Statistical Parity Difference: {metric.statistical_parity_difference()}")

History_of_SGA_FGR
0.0    0.32636
1.0    1.00000
Name: Preterm_birth, dtype: float64
Bias Metrics for History_of_SGA_FGR:
Disparate Impact: 0.3263598326359833
Statistical Parity Difference: -0.6736401673640167


In [ ]:
print(df.groupby('Placental_cord_insertion')['Preterm_birth'].mean())

# Ορισμός privileged και unprivileged groups
privileged_groups = [{'Placental_cord_insertion': 0}] # Κατηγορία 0 ως προνομιούχα
unprivileged_groups = [{'Placental_cord_insertion': 3}] # Κατηγορία 3 ως μη προνομιούχα

# Μετατροπή του dataset σε BinaryLabelDataset format
dataset = BinaryLabelDataset(
        df=df[['Placental_cord_insertion', 'Preterm_birth']].dropna(),
        label_names=['Preterm_birth'],
        protected_attribute_names=['Placental_cord_insertion']
    )

# Υπολογισμός των μετρικών bias
metric = BinaryLabelDatasetMetric(
        dataset,
        privileged_groups,
        unprivileged_groups
    )

print("Bias Metrics for Placental_cord_insertion:")
print(f"Disparate Impact: {metric.disparate_impact()}")
print(f"Statistical Parity Difference: {metric.statistical_parity_difference()}")

Placental_cord_insertion
0.0    0.089431
1.0    0.430285
2.0    0.272727
3.0    1.000000
Name: Preterm_birth, dtype: float64
Bias Metrics for Placental_cord_insertion:
Disparate Impact: 0.08943089430894309
Statistical Parity Difference: -0.9105691056910569


In [ ]:
print(df.groupby('Placental_cord_insertion_abnormal')['Preterm_birth'].mean())

# Ορισμός privileged και unprivileged groups στη σωστή μορφή
privileged_groups = {
    "Placental_cord_insertion_abnormal": [{'Placental_cord_insertion_abnormal': 1}]  # Κατηγορία 1 ως προνομιούχα
}

unprivileged_groups = {
    "Placental_cord_insertion_abnormal": [{'Placental_cord_insertion_abnormal': 0}]  # Κατηγορία 0 ως μη προνομιούχα
}

# Μετατροπή του dataset σε BinaryLabelDataset format
dataset = BinaryLabelDataset(
        df=df[['Placental_cord_insertion_abnormal', 'Preterm_birth']].dropna(),
        label_names=['Preterm_birth'],
        protected_attribute_names=["Placental_cord_insertion_abnormal"]
    )

# Υπολογισμός των μετρικών bias
metric = BinaryLabelDatasetMetric(
        dataset,
        privileged_groups=privileged_groups["Placental_cord_insertion_abnormal"],
        unprivileged_groups=unprivileged_groups["Placental_cord_insertion_abnormal"]
    )

print("Bias Metrics for Placental_cord_insertion_abnormal:")
print(f"Disparate Impact: {metric.disparate_impact()}")
print(f"Statistical Parity Difference: {metric.statistical_parity_difference()}")

Placental_cord_insertion_abnormal
0.0    0.338445
1.0    0.298246
Name: Preterm_birth, dtype: float64
Bias Metrics for Placental_cord_insertion_abnormal:
Disparate Impact: 1.1347851298241094
Statistical Parity Difference: 0.04019907380719051


In [ ]:
print(df.groupby("Gravida")["Preterm_birth"].mean())

privileged_groups = {
    "Gravida": [{"Gravida": x} for x in range(0, 3)],  # 0, 1, 2
}

unprivileged_groups = {
    "Gravida": [{"Gravida": x} for x in range(3, 8)],  # 3, 4, 5, 6, 7
}

# Μετατροπή σε BinaryLabelDataset
dataset = BinaryLabelDataset(df=df[['Gravida', 'Preterm_birth']].dropna(),
                                 label_names=['Preterm_birth'],
                                 protected_attribute_names=["Gravida"])

# Υπολογισμός των μετρικών bias
metric = BinaryLabelDatasetMetric(
        dataset,
        privileged_groups=privileged_groups["Gravida"],
        unprivileged_groups=unprivileged_groups["Gravida"]
    )

print("Bias Metrics for Gravida:")
print(f"Disparate Impact: {metric.disparate_impact()}")
print(f"Statistical Parity Difference: {metric.statistical_parity_difference()}")

Gravida
0.0    0.252955
1.0    0.376147
2.0    0.406015
3.0    0.490566
4.0    0.545455
5.0    0.300000
6.0    0.000000
7.0    1.000000
Name: Preterm_birth, dtype: float64
Bias Metrics for Gravida:
Disparate Impact: 1.500971345313259
Statistical Parity Difference: 0.16112781661264503


In [ ]:
print(df.groupby("Parity.1")["Preterm_birth"].mean())

privileged_groups = {
    "Parity.1": [{"Parity.1": x} for x in range(1, 6)]  # 1, 2, 3, 4, 5
}

unprivileged_groups = {
    "Parity.1": [{"Parity.1": 0}]  # 0
}

# Μετατροπή σε BinaryLabelDataset
dataset = BinaryLabelDataset(df=df[['Parity.1', 'Preterm_birth']].dropna(),
                                 label_names=['Preterm_birth'],
                                 protected_attribute_names=["Parity.1"])

# Υπολογισμός των μετρικών bias
metric = BinaryLabelDatasetMetric(
        dataset,
        privileged_groups=privileged_groups["Parity.1"],
        unprivileged_groups=unprivileged_groups["Parity.1"]
    )

print("Bias Metrics for Parity.1:")
print(f"  Disparate Impact: {metric.disparate_impact()}")
print(f"  Statistical Parity Difference: {metric.statistical_parity_difference()}")

Parity.1
0.0    0.314815
1.0    0.372881
2.0    0.354839
3.0    0.333333
4.0    0.333333
5.0    0.333333
6.0    1.000000
Name: Preterm_birth, dtype: float64
Bias Metrics for Parity.1:
  Disparate Impact: 0.855475040257649
  Statistical Parity Difference: -0.05318518518518517


In [ ]:
print(df.groupby("GDM")["Preterm_birth"].mean())

privileged_groups = {
    "GDM": [{"GDM": 0}],  # χαμηλότερο ποσοστό πρόωρων γεννήσεων
}

unprivileged_groups = {
    "GDM": [{"GDM": 1}],  # υψηλότερο ποσοστό πρόωρων γεννήσεων
}

# Μετατροπή σε BinaryLabelDataset
dataset = BinaryLabelDataset(df=df[['GDM', 'Preterm_birth']].dropna(),
                                 label_names=['Preterm_birth'],
                                 protected_attribute_names=["GDM"])

# Υπολογισμός των μετρικών bias
metric = BinaryLabelDatasetMetric(
        dataset,
        privileged_groups=privileged_groups["GDM"],
        unprivileged_groups=unprivileged_groups["GDM"]
    )

print("Bias Metrics for GDM:")
print(f"  Disparate Impact: {metric.disparate_impact()}")
print(f"  Statistical Parity Difference: {metric.statistical_parity_difference()}")

GDM
0.0    0.33475
1.0    0.37931
Name: Preterm_birth, dtype: float64
Bias Metrics for GDM:
  Disparate Impact: 1.1331143951833607
  Statistical Parity Difference: 0.04456007915277216


In [ ]:
print(df.groupby("PREECLAMPSIA")["Preterm_birth"].mean())

privileged_groups = {
    "PREECLAMPSIA": [{"PREECLAMPSIA": 0}]  # χαμηλότερο ποσοστό πρόωρων γεννήσεων
}

unprivileged_groups = {
    "PREECLAMPSIA": [{"PREECLAMPSIA": 1}]  # υψηλότερο ποσοστό πρόωρων γεννήσεων
}

# Μετατροπή σε BinaryLabelDataset
dataset = BinaryLabelDataset(df=df[['PREECLAMPSIA', 'Preterm_birth']].dropna(),
                                 label_names=['Preterm_birth'],
                                 protected_attribute_names=["PREECLAMPSIA"])

# Υπολογισμός των μετρικών bias
metric = BinaryLabelDatasetMetric(
        dataset,
        privileged_groups=privileged_groups["PREECLAMPSIA"],
        unprivileged_groups=unprivileged_groups["PREECLAMPSIA"]
    )

print("Bias Metrics for PREECLAMPSIA:")
print(f"Disparate Impact: {metric.disparate_impact()}")
print(f"Statistical Parity Difference: {metric.statistical_parity_difference()}")

PREECLAMPSIA
0.0    0.326702
1.0    0.933333
Name: Preterm_birth, dtype: float64
Bias Metrics for PREECLAMPSIA:
Disparate Impact: 2.856837606837607
Statistical Parity Difference: 0.6066317626527051


In [ ]:
print(df.groupby("b-hcg")["Preterm_birth"].mean())

# Βρίσκουμε το κατώφλι της διάμεσου
threshold = df["b-hcg"].median()

# Δημιουργία δυαδικής κατηγορίας
df["b-hcg_category"] = (df["b-hcg"] >= threshold).astype(int)  # 1 = Υψηλό, 0 = Χαμηλό

# Ορισμός των privileged και unprivileged groups
privileged_groups = [{'b-hcg_category': 0}]  # Χαμηλά επίπεδα
unprivileged_groups = [{'b-hcg_category': 1}]  # Υψηλά επίπεδα

# Δημιουργία BinaryLabelDataset
dataset = BinaryLabelDataset(
    df=df[['b-hcg_category', 'Preterm_birth']].dropna(),
    label_names=["Preterm_birth"],
    protected_attribute_names=["b-hcg_category"]
)

# Υπολογισμός των Bias Metrics
metric = BinaryLabelDatasetMetric(dataset,
                                  privileged_groups=privileged_groups,
                                  unprivileged_groups=unprivileged_groups)

# Εκτύπωση αποτελεσμάτων
print("Bias Metrics for b-hcg:")
print(f"Disparate Impact: {metric.disparate_impact()}")
print(f"Statistical Parity Difference: {metric.statistical_parity_difference()}")

b-hcg
0.20    1.000000
0.30    1.000000
0.40    1.000000
0.50    0.222222
0.56    0.000000
          ...   
1.70    0.200000
1.80    0.300000
1.90    0.066667
2.00    1.000000
2.10    1.000000
Name: Preterm_birth, Length: 91, dtype: float64
Bias Metrics for b-hcg:
Disparate Impact: 1.011945088251025
Statistical Parity Difference: 0.003990114045796966


In [ ]:
print(df.groupby("EFW")["Preterm_birth"].mean())
# Βρίσκουμε το κατώφλι της διάμεσου
threshold_efw = df["EFW"].median()

# Δημιουργία δυαδικής κατηγορίας για το EFW
df["EFW_category"] = (df["EFW"] >= threshold_efw).astype(int)  # 1 = Υψηλό, 0 = Χαμηλό

# Ορισμός των privileged και unprivileged groups
privileged_groups = [{'EFW_category': 1}]  # Υψηλό EFW (πιο προστατευμένο)
unprivileged_groups = [{'EFW_category': 0}]  # Χαμηλό EFW (πιθανώς υψηλότερος κίνδυνος)

# Δημιουργία BinaryLabelDataset
dataset = BinaryLabelDataset(
    df=df[['EFW_category', 'Preterm_birth']].dropna(),
    label_names=["Preterm_birth"],
    protected_attribute_names=["EFW_category"]
)

# Υπολογισμός των Bias Metrics
metric = BinaryLabelDatasetMetric(dataset,
                                  privileged_groups=privileged_groups,
                                  unprivileged_groups=unprivileged_groups)

# Εκτύπωση αποτελεσμάτων
print("Bias Metrics for EFW:")
print(f"Disparate Impact: {metric.disparate_impact()}")
print(f"Statistical Parity Difference: {metric.statistical_parity_difference()}")

EFW
527.0     1.0
653.0     1.0
664.0     1.0
716.0     1.0
754.0     1.0
         ... 
2543.0    1.0
2573.0    1.0
2739.0    0.0
2809.0    1.0
3258.0    1.0
Name: Preterm_birth, Length: 569, dtype: float64
Bias Metrics for EFW:
Disparate Impact: 1.1542170644001308
Statistical Parity Difference: 0.04813345747190939


In [ ]:
print(df.groupby("BW centile")["Preterm_birth"].mean())

# Βρίσκουμε το κατώφλι της διάμεσου
threshold_bw = df["BW centile"].median()

# Δημιουργία δυαδικής κατηγορίας για το BW centile
df["BW_category"] = (df["BW centile"] >= threshold_bw).astype(int)  # 1 = Υψηλό, 0 = Χαμηλό

# Ορισμός των privileged και unprivileged groups
privileged_groups = [{'BW_category': 1}]  # Υψηλό BW centile (πιο προστατευμένο)
unprivileged_groups = [{'BW_category': 0}]  # Χαμηλό BW centile (πιθανώς υψηλότερος κίνδυνος)

# Δημιουργία BinaryLabelDataset
dataset = BinaryLabelDataset(
    df=df[['BW_category', 'Preterm_birth']].dropna(),
    label_names=["Preterm_birth"],
    protected_attribute_names=["BW_category"]
)

# Υπολογισμός των Bias Metrics
metric = BinaryLabelDatasetMetric(dataset,
                                  privileged_groups=privileged_groups,
                                  unprivileged_groups=unprivileged_groups)

# Εκτύπωση αποτελεσμάτων
print("Bias Metrics for BW centile:")
print(f"Disparate Impact: {metric.disparate_impact()}")
print(f"Statistical Parity Difference: {metric.statistical_parity_difference()}")

BW centile
1.476402e-23    1.0
1.158687e-18    1.0
1.875135e-17    1.0
1.261376e-14    1.0
9.693321e-09    1.0
               ... 
9.988534e+01    0.0
9.992229e+01    0.0
9.997725e+01    0.0
9.998111e+01    0.0
9.999647e+01    0.0
Name: Preterm_birth, Length: 874, dtype: float64
Bias Metrics for BW centile:
Disparate Impact: 1.116883116883117
Statistical Parity Difference: 0.037113402061855705


In [17]:
print(df.groupby("Height")["Preterm_birth"].mean())

# Κατηγοριοποίηση
df['Height'] = df['Height'].apply(lambda x: 0 if x < 160 else 1)

# Ορισμός ομάδων
priv_height = [{'Height': 1}]  # Ψηλές ως προνομιούχες
unpriv_height = [{'Height': 0}]  # Κοντές ως μη προνομιούχες

print(df['Height'].value_counts())  # Πρέπει να δεις 0 και 1

# Δημιουργία BinaryLabelDataset
dataset_height = BinaryLabelDataset(
    df=df.dropna(subset=['Preterm_birth', 'Height']),
    label_names=['Preterm_birth'],
    protected_attribute_names=['Height']
)

# Υπολογισμός μετρικών bias
metric_height = BinaryLabelDatasetMetric(
    dataset_height,
    privileged_groups=priv_height,
    unprivileged_groups=unpriv_height
)

# Εκτύπωση αποτελεσμάτων
print("Bias Metrics for Height:")
print(f"Disparate Impact: {metric_height.disparate_impact()}")
print(f"Statistical Parity Difference: {metric_height.statistical_parity_difference()}")

Height
0    0.336082
Name: Preterm_birth, dtype: float64
Height
0    1956
Name: count, dtype: int64
Bias Metrics for Height:
Disparate Impact: nan
Statistical Parity Difference: nan


/usr/local/lib/python3.11/dist-packages/aif360/metrics/binary_label_dataset_metric.py:105: RuntimeWarning: invalid value encountered in scalar divide
  return (self.num_positives(privileged=privileged)


In [13]:
print(df['Height'].value_counts())
print(df.groupby('Height')['Preterm_birth'].value_counts(normalize=True))
df['Height'] = df['Height'].apply(lambda x: 1 if x < 160 else 0)
privileged_groups = [{'Height': 0}]
unprivileged_groups = [{'Height': 1}]
print(df['Preterm_birth'].unique())
print(df['Height'].unique())

Height
0    1956
Name: count, dtype: int64
Height  Preterm_birth
0       0.0              0.663918
        1.0              0.336082
Name: proportion, dtype: float64
[ 1.  0. nan]
[1]


In [20]:
print(df['Height'].describe())
print(df['Height'].isnull().sum())  # έλεγχος αν υπάρχουν NaNs

#Δημιουργία binary μεταβλητής με βάση το ύψος
df['Height'] = df['Height'].apply(lambda x: 1 if x < 160 else 0)

# Έλεγχος αν έγινε σωστά
print(df['Height'].value_counts())  # Πρέπει να υπάρχει και 0 και 1

# Αφαίρεση τυχόν NaNs
df = df.dropna(subset=['Height', 'Preterm_birth'])

# Ορισμός ομάδων
privileged_groups = [{'Height': 0}]  # Ύψος >=160
unprivileged_groups = [{'Height': 1}]  # Ύψος <160

dataset_height = BinaryLabelDataset(
    df=df,
    label_names=['Preterm_birth'],
    protected_attribute_names=['Height']
)

# Υπολογισμός fairness metrics
metric = BinaryLabelDatasetMetric(
    dataset_height,
    privileged_groups=privileged_groups,
    unprivileged_groups=unprivileged_groups
)

print("Bias Metrics for Height:")
print("Disparate Impact:", metric.disparate_impact())
print("Statistical Parity Difference:", metric.statistical_parity_difference())

count    970.0
mean       1.0
std        0.0
min        1.0
25%        1.0
50%        1.0
75%        1.0
max        1.0
Name: Height, dtype: float64
0
Height
1    970
Name: count, dtype: int64
Bias Metrics for Height:
Disparate Impact: nan
Statistical Parity Difference: nan


/usr/local/lib/python3.11/dist-packages/aif360/metrics/binary_label_dataset_metric.py:105: RuntimeWarning: invalid value encountered in scalar divide
  return (self.num_positives(privileged=privileged)
